# Aula 04 — Aplicação de um modelo ao projeto GaitDB

## Escolha do modelo

**Modelo:** Support Vector Machine (SVM) linear, usando estatísticas extraídas da série temporal de intervalos de passada por participante.

**Referência relacionada:** Joshi, D.; Khajuria, A.; Joshi, P. (2017). *An automatic non-invasive method for Parkinson's disease classification*. Computer Methods and Programs in Biomedicine, 145, 135–145. DOI: [10.1016/j.cmpb.2017.04.007](https://doi.org/10.1016/j.cmpb.2017.04.007). O artigo estuda SVM com características obtidas por wavelets a partir de variáveis de marcha, incluindo intervalo de passada.

**Importante:** Para manter a implementação compreensível e adequada ao tamanho muito pequeno do GaitDB, usamos três características simples por participante: média do intervalo, coeficiente de variação e autocorrelação de primeira ordem.

## Definição do problema

- **Tarefa:** classificação binária `parkinson` versus `controle` (jovens e idosos saudáveis juntos).
- **Unidade de análise:** participante. Cada pessoa gera uma única linha de características; não dividimos passadas da mesma pessoa entre treino e teste.
- **Separação:** 80% (12) dos participantes para treino e 20% (3) para teste, com semente fixa e estratificação pelos três grupos originais.

In [17]:
# ==============================================================================
# IMPORTAÇÕES DE BIBLIOTECAS E MÓDULOS
# ==============================================================================

import json
import math
from pathlib import Path
import joblib                            #Guardar e carregar modelos de Aprendizagem Automática (.joblib)
import numpy as np
import pandas as pd
import sklearn                           #Biblioteca principal de Machine Learning em Python
from IPython.display import display      #Exibir tabelas formatadas
from sklearn.impute import SimpleImputer #Preenche automaticamente valores em falta (NaN) na tabela

# Módulos de avaliação de desempenho e métricas do modelo
from sklearn.metrics import (
    accuracy_score,           # Acurácia simples (percentagem total de acertos)
    balanced_accuracy_score,  # Acurácia balanceada (média dos acertos por classe)
    confusion_matrix,         # Matriz de confusão (falsos/verdadeiros positivos e negativos)
    f1_score,                 # Pontuação F1 (média harmónica de precisão e sensibilidade)
    precision_score,          # Precisão (proporção de verdadeiros positivos nas predições)
    recall_score,             # Sensibilidade/Recall (proporção de positivos detetados)
)

#Função para dividir a base em conjuntos de treino e teste
from sklearn.model_selection import train_test_split

#Estrutura que encadeia pré-processamento e modelo num único fluxo
from sklearn.pipeline import Pipeline

# Padroniza as variáveis para média 0 e desvio-padrão 1 (z-score)
from sklearn.preprocessing import StandardScaler

#Support Vector Classifier (Modelo de IA baseado em Support Vector Machine)
from sklearn.svm import SVC


BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

ARQUIVO_SILVER = BASE / "data" / "silver" / "gaitdb_silver.parquet"
PASTA_RESULTADOS = BASE / "resultados"
PASTA_DOCS = BASE / "docs"

PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
PASTA_DOCS.mkdir(parents=True, exist_ok=True)

In [18]:
# ==============================================================================
# 2. CARREGAR A CAMADA SILVER
# ==============================================================================

# Lê a tabela já tratada diretamente do ficheiro Parquet da camada Silver
dados = pd.read_parquet(ARQUIVO_SILVER)

In [19]:
# ==============================================================================
# 3. EXTRAÇÃO DE CARACTERÍSTICAS POR PARTICIPANTE
# ==============================================================================

registros = []

# Agrupa todas as passadas individuais por participante e respetivo grupo
for (pid, grupo), parte in dados.groupby(["participante_id", "grupo"]):
    x = parte["intervalo_passada_(s)"].to_numpy(dtype=float)
    media = float(np.mean(x))                                         
    desvio = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0 
    cv = float(desvio / media) if media != 0 else np.nan                          # Coeficiente de Variação

    
    # Mede a dependência entre passos consecutivos)
    autocorr_lag1 = float(np.corrcoef(x[:-1], x[1:])[0, 1]) if len(x) >= 3 else np.nan


    # Adiciona o registo consolidado do participante
    registros.append({
        "participante_id": str(pid), 
        "grupo": grupo,                                                         # Grupo original (Jovem, Idoso, Parkinson)
        "rotulo": "parkinson" if grupo.lower() == "parkinson" else "controle",  # Rótulo binário
        "media_intervalo_s": media,                                             # Média
        "instab_caminhada": cv,                                                    # Irregularidade do passo
        "consistencia_ritmo_passos": autocorr_lag1                              # Ritmo sequencial
    })

# Cria a tabela final de caraterísticas com uma única linha por pessoa
features = pd.DataFrame(registros).sort_values("participante_id").reset_index(drop=True)
display(features)

,participante_id,grupo,rotulo,media_intervalo_s,instab_caminhada,consistencia_ritmo_passos
0,o1,idoso,controle,1.030661,0.017221,0.430142
1,o2,idoso,controle,1.167703,0.043174,0.021211
2,o3,idoso,controle,0.940825,0.032079,0.724085
3,o4,idoso,controle,0.956881,0.019217,0.274321
4,o5,idoso,controle,1.042140,0.015744,0.024876
5,pd1,parkinson,parkinson,1.250556,0.063832,0.300573
6,pd2,parkinson,parkinson,1.057805,0.047656,0.090549
7,pd3,parkinson,parkinson,1.369439,0.102566,0.234114
8,pd4,parkinson,parkinson,0.965625,0.082120,-0.036630
9,pd5,parkinson,parkinson,1.134138,0.036858,0.184452


In [20]:
# ==============================================================================
# 4. DIVISÃO EM TREINO E TESTE (POR PARTICIPANTE, SEM VAZAMENTO)
# ==============================================================================

SEMENTE = 42  # Valor fixo para garantir que a divisão é exatamente igual se voltar a rodar
FEATURES_MODELO = ["media_intervalo_s", "instab_caminhada", "consistencia_ritmo_passos"] 

# Divisão estratificada (80% treino, 20% teste) com base nas pessoas
treino, teste = train_test_split(
    features,
    test_size=0.20,
    random_state=SEMENTE,
    stratify=features["grupo"]
)

# Teste de segurança rigoroso: Garante 0 participantes partilhados entre treino e teste
assert len(set(treino["participante_id"]).intersection(set(teste["participante_id"]))) == 0

# Separação final entre variáveis independentes (X) e rótulo alvo (y)
X_treino, y_treino = treino[FEATURES_MODELO], treino["rotulo"]
X_teste, y_teste = teste[FEATURES_MODELO], teste["rotulo"]

In [21]:
# ==============================================================================
# 5. CONSTRUÇÃO E TREINAMENTO DO MODELO (PIPELINE)
# ==============================================================================

modelo = Pipeline(steps=[
    ("imputacao", SimpleImputer(strategy="median")),
    ("padronizacao", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0, class_weight="balanced"))
])

modelo.fit(X_treino, y_treino)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputacao', ...), ('padronizacao', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['controle','parkinson']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['media_intervalo_s','instab_caminhada','consistencia_ritmo_passos']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_valu

In [22]:
# ==============================================================================
# 6. INFERÊNCIA E MÉTRICAS DE AVALIAÇÃO
# ==============================================================================

# Executa as predições do modelo para o grupo de teste isolado
y_previsto = modelo.predict(X_teste)

# Obtém a pontuação de decisão (distância da fronteira do SVM)
escores = modelo.decision_function(X_teste)

# Tabela de resultados do teste
previsoes = teste[["participante_id", "grupo", "rotulo"]].copy()
previsoes = previsoes.rename(columns={"grupo": "grupo_real", "rotulo": "rotulo_real"})
previsoes["rotulo_previsto"] = y_previsto
previsoes["escore_decisao_svm"] = np.asarray(escores, dtype=float)          #Nível de certeza da IA (Negativo= Controle / Positivo = Parkinson)

# Função auxiliar para calcular o Intervalo de Confiança IC 95% (Método de Wilson)
def intervalo_wilson(sucessos, total, z=1.96):
    if total == 0: return (np.nan, np.nan)
    p = sucessos / total
    denominador = 1 + (z ** 2) / total
    centro = (p + (z ** 2) / (2 * total)) / denominador
    margem = (z * math.sqrt((p * (1 - p) + (z ** 2) / (4 * total)) / total)) / denominador
    return (max(0.0, centro - margem), min(1.0, centro + margem))

# Cálculo da matriz de confusão
matriz = confusion_matrix(y_teste, y_previsto, labels=["controle", "parkinson"])
tn, fp, fn, tp = matriz.ravel()

# Cálculo do quadro de métricas -> notas finais do robô em uma escala de 0.0 (pessimo) até 1.0 (perfeita))
metricas = pd.DataFrame([{        
    "n_teste": len(y_teste),
    "acuracia": accuracy_score(y_teste, y_previsto),
    "acuracia_balanceada": balanced_accuracy_score(y_teste, y_previsto),
    "sensibilidade_parkinson": recall_score(y_teste, y_previsto, pos_label="parkinson", zero_division=0),
    "especificidade_controle": recall_score(y_teste, y_previsto, pos_label="controle", zero_division=0),
    "f1_parkinson": f1_score(y_teste, y_previsto, pos_label="parkinson", zero_division=0),
}])

print("--- PREDIÇÕES DO TESTE ---")
display(previsoes)

print("--- MÉTRICAS ---")
display(metricas.round(3))

--- PREDIÇÕES DO TESTE ---


,participante_id,grupo_real,rotulo_real,rotulo_previsto,escore_decisao_svm
3,o4,idoso,controle,controle,-1.106497
7,pd3,parkinson,parkinson,parkinson,4.546202
14,y5,jovem,controle,controle,-1.188685


--- MÉTRICAS ---


,n_teste,acuracia,acuracia_balanceada,sensibilidade_parkinson,especificidade_controle,f1_parkinson
0,3,1.0,1.0,1.0,1.0,1.0


In [15]:
# ==============================================================================
# 7. SALVAMENTO DOS RESULTADOS E GERAÇÃO DO RELATÓRIO
# ==============================================================================

# Salva as tabelas de saída em CSV na pasta resultados/
previsoes.to_csv(PASTA_RESULTADOS / "predicoes_teste.csv", index=False)
metricas.to_csv(PASTA_RESULTADOS / "metricas_modelo.csv", index=False)

# Guarda o ficheiro binário do modelo treinado
joblib.dump(modelo, PASTA_RESULTADOS / "modelo_svm_gaitdb.joblib")

# Grava o ficheiro de metadados do modelo em formato JSON
metadados = {
    "nome_modelo": "Support Vector Classifier (SVC) linear",
    "versao_scikit_learn": sklearn.__version__,
    "caracteristicas": FEATURES_MODELO,
    "semente_divisao": SEMENTE
}
with open(PASTA_RESULTADOS / "modelo_svm_metadados.json", "w", encoding="utf-8") as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2)

# Gera o relatório resumido no formato Markdown para a pasta docs/
m = metricas.iloc[0]
relatorio = f"""# Relatório — Aplicação do Modelo ao GaitDB

## 1. Modelo e Tarefa
- Modelo: Support Vector Classifier (SVC Linear, C=1.0)
- Tarefa: Classificação binária (Parkinson vs. Controle)
- Unidade de análise: Participante

## 2. Resultados no Teste
- Acurácia Balanceada: {m['acuracia_balanceada']:.3f}
- Sensibilidade (Parkinson): {m['sensibilidade_parkinson']:.3f}
- Especificidade (Controle): {m['especificidade_controle']:.3f}

## 3. Limitações
- Amostra reduzida (15 participantes).
- Diferença nos protocolos de caminhada entre os grupos originais da fonte.
"""

# Escreve o relatório gerado no ficheiro .md final
(PASTA_DOCS / "relatorio_aplicacao_modelo.md").write_text(relatorio, encoding="utf-8")

print("Processo concluído com sucesso e ficheiros salvos!")

Processo concluído com sucesso e ficheiros salvos!


In [31]:
# ==============================================================================
# CONSULTA INTERATIVA POR ID DENTRO DO JUPYTER NOTEBOOK
# ==============================================================================

def consultar_participante_notebook():

    print("=== SISTEMA INTERATIVO DE CONSULTA (JUPYTER NOTEBOOK) ===")
    print("Exemplos de IDs válidos: pd1, pd2, pd3, o1, o2, y1, y2")
    
    # Exibe a caixa de texto no topo do VS Code / Jupyter
    id_digitado = input("\nDigite o ID do participante (ou 'sair' para encerrar): ")
    
    # Condição para encerrar sem executar busca
    if id_digitado.strip().lower() in ["sair", "exit", "0"]:
        print("\nConsulta encerrada.")
        return
    
    # Normaliza o texto para evitar erros de digitação (espaços ou maiúsculas)
    id_limpo = id_digitado.strip().lower()
    
    # Procura o ID na tabela 'features' que já existe na memória do Notebook
    dados_p = features[features["participante_id"].astype(str).str.lower() == id_limpo]
    
    # Valida se o participante existe
    if dados_p.empty:
        print(f"\n❌ Erro: O participante '{id_digitado}' não foi encontrado na base de dados.")
        print("   IDs disponíveis na memória:", ", ".join(features["participante_id"].tolist()))
        return
    
    # Seleciona as variáveis de entrada que o modelo utiliza
    features_modelo = ["media_intervalo_s", "instab_caminhada", "consistencia_ritmo_passos"]
    X_p = dados_p[features_modelo]
    
    # Executa a inferência com o modelo que já está treinado na memória
    predicao = modelo.predict(X_p)[0]
    escore = modelo.decision_function(X_p)[0]
    grupo_real = dados_p["grupo"].values[0] if "grupo" in dados_p.columns else "N/A"
    

    print("\n" + "=" * 55)
    print(f"       RESULTADO DA ANÁLISE: PARTICIPANTE '{dados_p['participante_id'].values[0]}'")
    print("=" * 55)
    print(f"• Grupo Real na Base      : {grupo_real}")
    print(f"• Classificação da IA     : {predicao.upper()}")
    print(f"• Escore de Decisão (SVM) : {escore:.4f}")
    
    if escore > 0:
        print("• Diagnóstico da IA       : Padrão compatível com PARKINSON")
    else:
        print("• Diagnóstico da IA       : Padrão compatível com CONTROLE (Saudável)")
    print("=" * 55 + "\n")


# Executa a função na célula
consultar_participante_notebook()

=== SISTEMA INTERATIVO DE CONSULTA (JUPYTER NOTEBOOK) ===
Exemplos de IDs válidos: pd1, pd2, pd3, o1, o2, y1, y2

       RESULTADO DA ANÁLISE: PARTICIPANTE 'y3'
• Grupo Real na Base      : jovem
• Classificação da IA     : CONTROLE
• Escore de Decisão (SVM) : -1.3430
• Diagnóstico da IA       : Padrão compatível com CONTROLE (Saudável)

